<a href="https://colab.research.google.com/github/djezmartin/GE120---UPD-Study-Spaces-Data/blob/main/AZANZA_CABILES_MARTINEZ_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Script Docstrings

In [ ]:
'''
GE 120 Final Project
Waze to Study: A Guide to UPD Study Spaces and Routes

    This script implements an interactive web-based map designed specifically to help UP Diliman
    students locate, filter, and navigate their way to different study spaces around the campus.
    It contains six (5) classes: StudySpace, Library, Cafe, StudyReadingHall, and CampusManager.

    StudySpace
    ========================================
    Represents a single study space on the UP Diliman campus.

    Methods:
      get_info(): Returns a formatted string with the study space's details
      get_distance(): Calculates the distance between the user's location and this study space using the Haversie formula
    ========================================

    Library
    ========================================
    A study space that is specifically a Library.
    ========================================

    Cafe
    ========================================
    A study space that is specifically a Cafe. Inherits from StudySpace and adds menu, price_range, and rating.

    Methods:
      get_info(): Overrides the parent get_info() to also show the menu
    ========================================

    StudyReadingHall
    ========================================
    A study space that is a Study/Reading Hall. Inherits from StudySpace without additional information.
    ========================================

    CampusManager
    ========================================
    Manages all the StudySpace objects.

    Methods:
         load_from_github(): Reads the CSV file and converts rows into objects
         filter_spaces(criteria, include_food=False): Filters the study spaces based on amenities and food availability
         get_all(): Returns all loaded study spaces
         get_nearest(user_lat, user_lng, n=5): Calculates distance for all spaces using the get_distance method, sorts them ascendingly, and returns the closest spaces.
         get_by_type(space_type): Filters spaces based on their type attribute (e.g., 'Library', 'Cafe', 'Study/Reading Hall')
         summary(): Prints a summary of the number of study spaces, libraries, cafes, and reading halls
    ========================================
'''

# Download and Imports

In [ ]:
!pip install openrouteservice --upgrade

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown
import math
import io
import requests
import folium
import openrouteservice

# INSTRUCTIONS

In [ ]:
with open('readme.txt', 'w+') as readme:
  description = ['Instructions for using Waze to Study: A Guide to UPD Study Spaces and Routes',
                 '\n  This "Waze to Study" study space finder is an interactive map built using Folium.',
                 '\n  It filters study enviornments within the campus using user location and selected amenities.',
                 '\n  It also uses an external routing API to generate navigation paths between user location and selected study space.',
                 '\n  The steps are as follows:',
                 '\n     1. Run the entire Python script to initialize the system',
                 '\n     2. Select preferred amenities in the checkbox',
                 '\n     3. Input user location coordinates',
                 '\n     4. Click on the filter button to display resulting study spaces',
                 '\n     5. From the results, select any of the filtered space then click Route to the chosen place',
                 '\n     6. Follow the generated route to get to your destination.\n',


                 '\nData Updates: ',
                 '\n- The study space (CSV dataset) is stored externally in Github.',
                 '\n- On every program run, the CSV is fetched and reloaded using the raw file link.',
                 '\n- Any changes made to the CSV are automatically reflected.\n',


                 '\nClass Interaction: ',
                 '\n- StudySpace stores key attributes (name, location, coordinates, type, services, and service hours).',
                 '\n- Subclasses (Library/Cafe/Study Hall) define specific space types.',
                 '\n- CampusManager processes user-selected amenities.',
                 '\n- StudySpace objects are generated into Folium markers.',
                 '\n- Routing module (OpenRouteService) calls an external API for navigation paths.\n',


                 '\nVisualization: ',
                 '\n- Built using Folium.',
                 '\n- Each study space is displayed as clickable marker.',
                 '\n- Filters dynamically update visible markers.',
                 '\n- Routing API generates path from user location to destination.',
                 '\n- Visualization is rendered directly in the Python environment (no HTML export required).'
                 ]

  readme.writelines(description)

# Classes

Creates the main data structures using parent class and child class inheritance. **`StudySpace`** acts as the base or parent. **`Library`**, **`Cafe`**, and **`StudyReadingHall`** inherit from the parent class, with the Cafe class extending it to handle specific attributes namely menu, price range, and ratings.

## Class 1: StudySpace

In [ ]:
class StudySpace:
  '''
  Represents a single study space on the UP Diliman campus.

  Attributes:
    name(str, constructor): The full name of the study space
    lat(float, constructor): GPS latitude coordinate
    lng(float, constructor): GPS longitude coordinate
    space_type(str, constructor): 'Library', 'Cafe', or 'Study/Reading Hall'
    services(str, constructor): Comma-separated list of available amenities
    hours(str, constructor): Service / operating hours
    distance(float): Distance from user location to study space; Initially None

  Methods:
    get_info: Returns a formatted string with the study space's details
    get_distance: Calculates the distance between the user's location and this study space using the Haversie formula
                      (formula for the distance on a sphere)
  '''

  def __init__(self, name, lat, lng, space_type, services="", hours=""):
    '''
    Builds the StudySpace object

    Args:
      name(str, constructor): The full name of the study space
      lat(float, constructor): GPS latitude coordinate
      lng(float, constructor): GPS longitude coordinate
      space_type(str, constructor): 'Library', 'Cafe', or 'Study/Reading Hall'
      services(str, constructor): Comma-separated list of available amenities
      hours(str, constructor): Service / operating hours
      distance(float): Distance from user location to study space; Initially None
    '''
    self.name = name
    self.lat = lat
    self.lng = lng
    self.space_type = space_type
    self.services = services
    self.hours = hours
    self.distance = None

  def get_info(self):
    '''
    Returns a formatted string with the study space's details.

    Returns:
      <str>: String formatting of the details of the StudySpace
    '''

    return (f"{self.name}\n"
            f"Type: {self.space_type}\n"
            f"Hours: {self.hours}\n"
            f"Coordinates: ({self.lat}, {self.lng})\n"
            f"Distance: {self.distance:.2f} meters\n"
            f"Services: {self.services}")

  def get_distance(self, user_lat, user_lng):
    '''
    Calculates the distance between the user's location and this study space using the Haversine Formula (formula for the distance)
    on a sphere.

    Args:
      user_lat(float): User's current GPS latitude
      user_lng(float): User's current GPS longitude

    Returns:
      <float>: Distance in meters
    '''
    R = 6371000 # Radius of the Earth in meters

    # Convert degrees to radians
    lat1 = math.radians(user_lat)
    lat2 = math.radians(self.lat)
    dlat = math.radians(self.lat - user_lat)
    dlng = math.radians(self.lng - user_lng)

    # The Haversine formula
    a = (math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlng / 2) ** 2)

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    self.distance = R * c  # Stores distance in meters
    return self.distance

## Class 2: Library

In [ ]:
class Library(StudySpace):
  '''
  A study space that is specifically a Library.

  Attributes:
    college(str, constructor): The college of the study space (e.g., "College of Human Kinetics")
  '''

  def __init__(self, name, lat, lng, services="", hours="", college=""):
    '''
    Builds the Library object

    Args:
      college(str, constructor): The college of the study space (e.g., "College of Human Kinetics")
    '''
    super().__init__(name, lat, lng, space_type="Library", services=services, hours=hours)
    self.college = college

## Class 3: Cafe

In [ ]:
class Cafe(StudySpace):
  '''
  A study space that is specifically a Cafe. Inherits from StudySpace and adds menu, price_range, and rating.

  Attributes:
    menu(str, constructor): General menu items of the cafe
    price_range(int, constructor): Price range of the items in the menu
    rating(float, constructor): Rating of the cafe, based from Google Maps

  Methods:
    get_info: Overrides the parent get_info() to also show the menu
  '''

  def __init__(self, name, lat, lng, services="", hours="", menu="", price_range="", rating=None):
    '''
    Builds the Cafe object

    Args:
      menu(str, constructor): General menu of the cafe
      price_range(int, constructor): Price range of the items in the menu
      rating(float, constructor): Rating of the cafe, based from Google Maps
    '''
    super().__init__(name, lat, lng, space_type="Cafe", services=services, hours=hours)

    self.menu = menu
    self.price_range = price_range
    self.rating = rating  # None for cafes with no reviews

  def get_info(self):
    '''
    Overrides parent get_info() to also show rating and menu.

    Returns:
      cafe_info(str): String formatting of the details of the Cafe; same as StudySpace info but with ratings
    '''
    cafe_info = (super().get_info() +
                f"Rating: {self.rating}\5.0\n"
                f"Menu: {self.menu}\n"
                f"Price Range: {self.price_range}")

    return cafe_info

## Class 4: StudyReadingHall

In [ ]:
class StudyReadingHall(StudySpace):
  '''
  A study space that is a Study/Reading Hall. Inherits from StudySpace without additional attributes or methods.
  '''

  def __init__(self, name, lat, lng, services="", hours=""):
    '''
    Builds the StudyReadingHall object
    '''
    super().__init__(name, lat, lng, space_type="Study/Reading Hall", services=services, hours=hours)

# Data Processing

Handles and manages the study space data necessary for the application. Its designed to process the retrieval of the raw CSV file from GitHub, the cleaning of the text, and the convertion of each entry into usable objects. The logic needed for filtering these locations based on user preference is also included.

## Class 5: CampusManager

In [ ]:
class CampusManager:
  '''
  Manages all the StudySpace objects.

  Attributes:
    url(str, constructor): GitHub link to the CSV data file
    spaces(list): Storage list containing all StudySpace objects

  Methods:
    load_from_github: Reads the CSV file and converts rows into objects
    filter_spaces: Filters the study spaces based on amenities and food availability
    get_all: Returns all loaded study spaces
    get_nearest: Calculates distance for all spaces using the get_distance method, sorts them ascendingly, and returns the closest spaces.
    get_by_type: Filters spaces based on their type attribute (e.g., 'Library', 'Cafe', 'Study/Reading Hall')
    summary: Prints a summary of the number of study spaces, libraries, cafes, and reading halls
  '''

  def __init__(self, url):
    '''
    Builds the CampusManager object

    Args:
      url(str, constructor): GitHub link to the CSV data file
      spaces(list): Storage list containing all StudySpace objects
    '''
    self.url = url
    self.spaces = []

    # Initialized here for testing
    self.load_from_github()

  def load_from_github(self):
    '''
    Loads the CSV files from GitHub directly without the need to upload the files manually every time
    '''
    self.spaces = []

    for url in self.url:
      try:
        # Using requests to get data from web; Source: https://requests.readthedocs.io/en/latest/
        response = requests.get(url)
        response.raise_for_status()

        # Convert web response to dataframe
        df = pd.read_csv(io.StringIO(response.text))

        # Cleans and extracts raw values from each row
        for _, row in df.iterrows():
          name     = str(row['Name']).strip()
          space_type = str(row['Classification']).lower().strip()
          lat      = float(row['Lat'])
          lng      = float(row['Lng'])
          raw_services = str(row['Services Offered']).strip()

          services = [
              service.lower()
              .replace("_", "")
              .replace("-", "")
              .replace(" ", "")
              .strip()

            for service in raw_services
                        .replace(";", ",")
                        .split(",")
          ]

          hours = str(row['Service Hours']).strip()

          # Processses libraries
          if "library" in space_type:
            college = str(row.get("College", "")).strip()
            space = Library(name, lat, lng, services=services, hours=hours, college=college)

          # Processses cafes
          elif "cafe" in space_type:
            menu = str(row.get("Menu", "")).strip()
            price_range = str(row.get("Price Range", "")).strip()
            raw_rating = row.get("Ratings", None)

            # Converts the rating to float
            try:
                rating = float(raw_rating)
            except (ValueError, TypeError):
                rating = None

            space = Cafe(name, lat, lng, services=services, hours=hours, menu=menu, price_range=price_range, rating=rating)

          # Processses study hall
          else:
            space = StudyReadingHall(name, lat, lng, services=services, hours=hours)

          self.spaces.append(space)

        print(f"Total spaces loaded: {len(self.spaces)}")

      except Exception as e:
        print(f"Error loading {url}: {e}")

  def filter_spaces(self, criteria, include_food=False):
    '''
    Filters study spaces based on amenities and food availability

    Args:
      criteria(list): Keywords representing required amenities
      include_food(bool): True for cafes, false for libraries or study/reading halls

    Returns:
      results(list): A list of study spaces that match the filters
    '''

    results = []

    # Fix criteria formatting
    normalized_criteria = [
        criterion.lower()
                 .replace("_", "")
                 .replace("-", "")
                 .replace(" ", "")
        for criterion in criteria
    ]

    for space in self.spaces:
        # Filters food amenity
        if include_food:
            if not isinstance(space, Cafe):
                continue

        else:
            if isinstance(space, Cafe):
                continue

        # Checks if all amenities exist
        matches_all = all(
            criterion in space.services
            for criterion in normalized_criteria)

        # Appends resulting spaces
        if matches_all:
            results.append(space)

    return results

  def get_nearest(self, spaces, user_lat, user_lng):
    '''
    Calculates the distance of each study from the user's location and sorts the
    spaces from nearest to farthest.

    Args:
      spaces(list): A list of StudySpace objects to be ranked by distance
      user_lat(float): User's current GPS latitude
      user_lng(float): User's current GPS longitude

    Returns:
      nearest_spaces(list): A list of StudySpace objects sorted by ascending distance
    '''
    nearest_spaces = []

    for space in spaces:
      space.get_distance(user_lat, user_lng)

    nearest_spaces = sorted(spaces, key=lambda s: s.distance)

    return nearest_spaces

  def get_all(self):
    '''
    Returns all loaded study spaces

    Returns:
      <list>: A list containing all StudySpace Objects
    '''
    return self.spaces

  def get_by_type(self, space_type):
    '''
    Filters spaces based on their type attribute (e.g., 'Library', 'Cafe', 'Study/Reading Hall')

    Args:
      space_type(str): The study space type to filter (e.g., 'Library', 'Cafe', 'Study/Reading Hall')

    Returns:
      filtered_spaces(list): A list of StudySpace objects matching the specified type
    '''
    filtered_spaces = []

    for space in self.spaces:
      if space.space_type.lower() == space_type.lower():
          filtered_spaces.append(space)

    return filtered_spaces

  def summary(self):
    '''
    Prints a summary of the number of study spaces, libraries, cafes, and reading halls.
    '''
    # Initializes the counters
    libraries = 0
    cafes = 0
    halls = 0

    # Counts each space type
    for s in self.spaces:
      if isinstance(s, Library):
        libraries += 1
      elif isinstance(s, Cafe):
          cafes += 1
      else:
          halls += 1

    # Displays the final counts
    print("=" * 40)
    print(f"Libraries: {libraries}\n"
          f"Cafes: {cafes}\n"
          f"Study Halls: {halls}")

In [ ]:
#URL for CSV file
my_github_links = [
    "https://raw.githubusercontent.com/djezmartin/GE120---UPD-Study-Spaces-Data/refs/heads/main/Study%20Spaces.csv"]

In [ ]:
# Initializes, loads, and summarizes the data
manager = CampusManager(my_github_links)
manager.summary()

Total spaces loaded: 68
Libraries: 52
Cafes: 13
Study Halls: 3


# Visualization - Processing

Uses API's openrouteservice to show the route from the user's location to the study space.

***Changes since final presentation:***

1. Different icons for different study spaces
2. Added the folium.LatLngPopup for easy input of coordinates

In [ ]:
# API Key
api_key = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImE4MjRkY2IwMDk5NzRjNmJhNWY1ZmM1YTNhODI2NzM4IiwiaCI6Im11cm11cjY0In0="

# OpenRouteService client
client = openrouteservice.Client(key=api_key)

# Default starting point centered at UPD
user_loc = [14.655216, 121.068091]
results = []

# Function for building map
def map_building(spaces, user_location, selected_space=None):
  '''
  Generates an interactive Folium map populated with user and study space location markers.

  Args:
    spaces(list): A list of study space objects (e.g., Library, Cafe, StudyReadingHall)
    user_location(tuple): The latitude and longitude coordinates of the user's current location
    selected_space(StudySpace): A specific study space object to highlight or focus on (defaults to None)

  Returns:
    built_map(folium.Map): The configured interactive map object with all location markers added
  '''
  #Base map
  built_map = folium.Map(location=user_location, zoom_start=15)

  # Add a popup of (lat, long) everytime you click at the folium map for copy-pasting to input boxes
  built_map.add_child(folium.LatLngPopup())

  # User Marker
  folium.Marker(user_location, popup='You are here.',
                icon=folium.Icon(color='red', icon='user'),
                ).add_to(built_map)

  # Loop through spaces
  for space in spaces:
    marker_color = 'blue'
    popup_content = f"""
                    <b>{space.name}</b><br><br>

                    Type: {space.space_type}<br>
                    Distance: {space.distance:.2f} meters<br>
                    Hours: {space.hours}<br>
                    Amenities: {space.services}
                    """

    # For libraries
    if isinstance(space, Library):
      marker_color = 'lightblue'
      marker_icon = 'book'
      marker_prefix = 'glyphicon'
      popup_content = f"""
      <b>{space.name}</b><br><br>

      Type: {space.space_type}<br>
      Distance: {space.distance:.2f} meters<br>
      Hours: {space.hours}<br>
      Amenities: {space.services}
      """

    # For cafes
    elif isinstance(space, Cafe):
      marker_color = 'orange'
      marker_icon = 'coffee'
      marker_prefix = 'fa'
      popup_content = f"""
      <b>{space.name}</b><br><br>

      Type: {space.space_type}<br>
      Distance: {space.distance:.2f} meters<br>
      Hours: {space.hours}<br>
      Amenities: {space.services}<br>
      Rating: {space.rating}/5<br>
      Menu: {space.menu}<br>
      Price Range: {space.price_range}
      """

    # For study halls
    elif isinstance(space, StudyReadingHall):
      marker_color = 'lightgreen'
      marker_icon = 'glasses'
      marker_prefix = 'fa'

    # Add markers
    folium.Marker(
        [space.lat, space.lng],
        popup=folium.Popup(popup_content, max_width=300),
        icon=folium.Icon(color=marker_color, icon=marker_icon, prefix=marker_prefix)
    ).add_to(built_map)

    # Route for selected study space
    if selected_space == space:
      coords = [
          [user_location[1], user_location[0]],
          [space.lng, space.lat]
      ]

      #API's openrouteservice routing
      try:
        route = client.directions(
            coordinates=coords,
            profile='foot-walking',
            format='geojson'
        )

        # Get the route geometry from the API response
        geometry = route['features'][0]['geometry']

        # Route path line
        folium.GeoJson(
            geometry,
            style_function=lambda x: {
                'color': 'blue',
                'weight': 5,
                'opacity': 0.8
            }
        ).add_to(built_map)

      except Exception as e:
        print(f"Route failed for {space.name}")
        print(e)

  #Displays the map with the user's pinned location and route to the study space
  return built_map

# Inputs (Widgets, Text), Visualization

Sets up interactive buttons for the user interface. It creates the check boxes and text fields designed to let the user select a study space based on their preferres amenities or services.

The map showing the study spaces and the route to the selected study space is also shown here.

***Changes since final presentation:***

1. Added a simple User Interface
2. The results after clicking filter is ranked from nearest to farthest from user (based on the Haversine formula)
3. Cleaned the map output; a single working folium map is generated instead of two

In [ ]:
# Widgets
display(Markdown('## Waze to Study: Your UPD Study Space Finder'))

wifi = widgets.Checkbox(
    value=False,
    description='Wi-Fi'
)

reading = widgets.Checkbox(
    value=False,
    description='Reading Area'
)

photocopy = widgets.Checkbox(
    value=False,
    description='Photocopy'
)

computers = widgets.Checkbox(
    value=False,
    description='Computers'
)

outlets = widgets.Checkbox(
    value=False,
    description='Outlets'
)

food = widgets.Checkbox(
    value=False,
    description='Food'
)

user_lat = widgets.FloatText(
    description='Latitude:'
)

user_lng = widgets.FloatText(
    description='Longitude:'
)

button = widgets.Button(
    description='Filter',
    color='green'
)

# Default value (placeholder); center of UPD
user_lat = widgets.FloatText(description='Latitude:', value=14.655216)
user_lng = widgets.FloatText(description='Longitude:', value=121.068091)

# Output widgets for text and map
results_output = widgets.Output()
map_output = widgets.Output()

# UI - left sidebar
sidebar_controls = widgets.VBox(
    [widgets.Label(value="Filter Amenities:"), wifi, reading, photocopy, computers, outlets, food,
     widgets.Label(value="Coordinates:"), user_lat, user_lng, button],
    layout=widgets.Layout(width='240px', padding='10px', border='1px solid #ccc', margin='0 10px 0 0')
)

# Overall UI
dashboard_layout = widgets.HBox([
    sidebar_controls,
    widgets.VBox([results_output], layout=widgets.Layout(width='340px', max_height='550px', overflow_y='auto')),
    widgets.VBox([map_output], layout=widgets.Layout(flex='1 1 auto', height='550px'))
])
display(dashboard_layout)

# Load the initial map at startup; calls map_building w/ user's default pinned location and the clickable LatLngPopup for copy and paste
with map_output:
    display(map_building([], [14.655216, 121.068091], selected_space=None))

# Filtering for study spaces' selected amenities
def filter_data(b):
  '''
  Filters and displays study spaces based on selected amenity widgets and user location.

  Parameters:
    b(ipywidgets.Button): The button widget instance that triggered the click event
  '''

  # Clear previous text results and map display before rendering new filtered data
  results_output.clear_output()
  map_output.clear_output()

  # Stores selected amenities
  selected = []

  if wifi.value:
    selected.append('wifi')

  if reading.value:
    selected.append('readingarea')

  if photocopy.value:
    selected.append('photocopy')

  if computers.value:
    selected.append('computers')

  if outlets.value:
    selected.append('outlets')

  # Location of user (saves as a global location variable)
  global user_loc
  user_loc = (user_lat.value, user_lng.value)

  # Saves results globally
  global results

  # Filter results; calls manager (variable for CampusManager)
  results = manager.filter_spaces(selected, include_food=food.value)
  results = manager.get_nearest(results, user_lat.value, user_lng.value)

  # Display results
  with results_output:
    print("Matching Study Spaces")
    print("=" * 40)

    # No results
    if len(results) == 0:
      print("No matching spaces found.")

    else:
      for space in results:
        print(space.get_info())
        print("-" * 40)

        # Route button
        route_button = widgets.Button(
            description=f'Route to {space.name}',
            button_style='success'
            )

        # Callback function
        def make_callback(s=space):
            '''
            Creates a callback function for a button click event to display a built map.

            Parameters:
            s(StudySpace): The specific study space to highlight on the map (defaults to space)

            Returns:
            callback(function): A function that clears and updates the map output widget when triggered
            '''
            def callback(btn):
                '''
                Handles route button click events to update the dashboard map display.

                Clears the previous map visualization canvas, recaptures the current user
                coordinates, and renders a new Folium map overlay featuring the active walking
                route path to the targeted study space destination.

                Parameters:
                -----------
                btn : widgets.Button
                    The specific clicked ipywidget button object instance triggering the event.
                '''
                #Used again so that only a single folium map is generated (previously; a new one is created at the bottom of the empty map)
                map_output.clear_output()

                # Map rendering with the route
                with map_output:
                    output_map = map_building(results, user_loc, selected_space=s)
                    display(output_map)

            return callback

        # Route to selected study space button
        route_button.on_click(make_callback())
        display(route_button)

    # Show all matching pins right away(except cafes) when clicking Filter without checking any boxes
    with map_output:
        initial_map = map_building(results, user_loc, selected_space=None)
        display(initial_map)

button.on_click(filter_data)

## Waze to Study: Your UPD Study Space Finder

# References

In [ ]:
'''
Those topics unfamiliar to the students and not taught in class were gathered online.

References:
    Integrating API: Amazon Web Services. (2025). What is an API? - API Beginner’s Guide - AWS. Amazon Web Services, Inc. https://aws.amazon.com/what-is/api/
    Integrating API's openrouteservice: OpenRouteService route planner - directions, isochrones and places. (n.d.). https://maps.openrouteservice.org/
    GeoJSON text: Dashboard | ORS. (n.d.). https://openrouteservice.org/dev/#/api-docs/v2/directions/%7Bprofile%7D/post
    Widgets: Simple Widget Introduction — Jupyter Widgets 8.1.8 documentation. (n.d.). https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Basics.html
'''

## [Github link](https://github.com/djezmartin/GE120---UPD-Study-Spaces-Data/tree/main)